# Water Quality ML Model — 6 Parameter Retrain
Retraining the water quality classifier on the **6 parameters that match the `lab_tests` database table**:
`ph`, `turbidity`, `dissolved_oxygen`, `nitrates`, `phosphates`, `salinity`

Outputs three artifacts saved to `model/artifacts/`:
- `model.pkl` — trained Random Forest pipeline
- `imputer.pkl` — median imputer for missing values
- `label_encoder.pkl` — risk level encoder

## 1. Install & Import

In [ ]:
%pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("All libraries imported successfully.")

All libraries imported successfully.


: 

## 2. Load Dataset
Upload `water_samples_renamed.xlsx` to Colab or update the path below.

In [ ]:
df_raw = pd.read_excel("/content/water_samples_renamed.xlsx")
print(f"Raw dataset shape: {df_raw.shape}")
print(f"\nColumns available:\n{list(df_raw.columns)}")

## 3. Map to `lab_tests` Column Names
Rename the dataset columns to exactly match the database table.

In [ ]:
# Mapping: original dataset column → lab_tests column name
COL_MAP = {
    'pH':           'ph',
    'conductivity': 'salinity',          # conductivity used as salinity proxy
    'DO':           'dissolved_oxygen',
    'NO3':          'nitrates',
    'PO4':          'phosphates',
}

df = df_raw[list(COL_MAP.keys())].copy()
df.rename(columns=COL_MAP, inplace=True)

# turbidity is accepted from the frontend at inference time
# but is not present in the training dataset
# we include it as a 0.0 placeholder so the imputer learns its median = 0
df['turbidity'] = 0.0

FEATURES = ['ph', 'turbidity', 'dissolved_oxygen', 'nitrates', 'phosphates', 'salinity']

print("Renamed columns:", list(df.columns))
print(f"\nDataset shape: {df.shape}")
df.head()

## 4. Explore the Data

In [ ]:
print("=== Basic Info ===")
df.info()

In [ ]:
print("=== Descriptive Statistics ===")
df[FEATURES].describe().round(3)

In [ ]:
print("=== Missing Values ===")
missing = df[FEATURES].isnull().sum()
pct     = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing': missing, 'Pct (%)': pct})

In [ ]:
print("=== Skewness (guides imputation strategy) ===")
df[FEATURES].skew().round(2)

In [ ]:
# Distribution plots for all 6 features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = ['#3498DB','#2ECC71','#E74C3C','#F39C12','#9B59B6','#1ABC9C']
for i, feat in enumerate(FEATURES):
    axes[i].hist(df[feat].dropna(), bins=30, color=colors[i], edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions (Training Data)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. WQI Calculation
Using the same formula and WHO/EPA standards as the production `wqi_engine.py`.
Column names match `lab_tests` exactly.

In [ ]:
# WHO/EPA weights for the 6 lab_tests parameters
WEIGHTS = {
    'ph':               0.22,
    'turbidity':        0.18,
    'dissolved_oxygen': 0.25,
    'nitrates':         0.15,
    'phosphates':       0.10,
    'salinity':         0.10,
}

# WHO/EPA limits
LIMITS = {
    'ph':               {'mode': 'ph',      'ideal': 7.0,  'min': 6.5,  'max': 8.5},
    'turbidity':        {'mode': 'normal',  'ideal': None, 'min': None, 'max': 4.0},
    'dissolved_oxygen': {'mode': 'inverse', 'ideal': 14.6, 'min': 6.5,  'max': 14.6},
    'nitrates':         {'mode': 'normal',  'ideal': None, 'min': None, 'max': 50.0},
    'phosphates':       {'mode': 'normal',  'ideal': None, 'min': None, 'max': 0.1},
    'salinity':         {'mode': 'normal',  'ideal': None, 'min': None, 'max': 1500},
}

def calc_sub_index(param, value):
    """Calculate quality sub-index for a single parameter (0=best, 100=worst)."""
    if pd.isna(value):
        return np.nan
    lim = LIMITS[param]
    if lim['mode'] == 'ph':
        dev     = abs(value - lim['ideal'])
        max_dev = max(abs(lim['min'] - lim['ideal']), abs(lim['max'] - lim['ideal']))
        return float(np.clip(dev / max_dev * 100, 0, 100))
    elif lim['mode'] == 'inverse':
        if value >= lim['ideal']: return 0.0
        return float(np.clip((lim['ideal'] - value) / lim['ideal'] * 100, 0, 100))
    else:
        return float(np.clip(value / lim['max'] * 100, 0, 100))

def calc_wqi(row):
    """Weighted WQI across all 6 parameters."""
    total_w = weighted_sum = 0
    for param, w in WEIGHTS.items():
        qi = calc_sub_index(param, row[param])
        if not np.isnan(qi):
            weighted_sum += qi * w
            total_w      += w
    return round(weighted_sum / total_w, 2) if total_w > 0 else np.nan

df['WQI Score'] = df.apply(calc_wqi, axis=1)
print(f"WQI calculated for {df['WQI Score'].notna().sum()} / {len(df)} rows")
print(f"\nWQI Score stats:")
print(df['WQI Score'].describe().round(2))

## 6. Health Score
Non-linear banding — same formula as `wqi_engine.py`.

In [ ]:
def calc_health_score(wqi):
    if pd.isna(wqi):        return np.nan
    if wqi <= 25:           return round(100 - wqi * 0.4, 1)          # 100–90%
    if wqi <= 50:           return round(90  - (wqi - 25)  * 1.2, 1)  # 90–60%
    if wqi <= 75:           return round(60  - (wqi - 50)  * 1.2, 1)  # 60–30%
    if wqi <= 100:          return round(30  - (wqi - 75)  * 0.8, 1)  # 30–10%
    return round(max(0, 10 - (wqi - 100) * 0.1), 1)

df['Health Score (%)'] = df['WQI Score'].apply(calc_health_score)

print("Health Score stats:")
print(df['Health Score (%)'].describe().round(2))

## 7. Assign Risk Levels

In [ ]:
def assign_risk(wqi):
    if pd.isna(wqi):   return 'Unknown'
    if wqi <= 25:      return 'Low Risk'
    if wqi <= 50:      return 'Moderate Risk'
    if wqi <= 75:      return 'High Risk'
    return 'Critical Risk'

df['Risk Level'] = df['WQI Score'].apply(assign_risk)

print("Risk Level distribution:")
print(df['Risk Level'].value_counts())
print(f"\nRows with 'Unknown' risk (will be dropped): {(df['Risk Level'] == 'Unknown').sum()}")

In [ ]:
# WQI vs Health Score scatter coloured by risk level
risk_colors = {
    'Low Risk': '#2ECC71', 'Moderate Risk': '#F1C40F',
    'High Risk': '#E67E22', 'Critical Risk': '#E74C3C', 'Unknown': '#BDC3C7'
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for risk, color in risk_colors.items():
    subset = df[df['Risk Level'] == risk]
    axes[0].scatter(subset['WQI Score'], subset['Health Score (%)'],
                    c=color, label=risk, alpha=0.7, s=40)
axes[0].set_xlabel('WQI Score')
axes[0].set_ylabel('Health Score (%)')
axes[0].set_title('WQI vs Health Score by Risk Level')
axes[0].legend()

df['Risk Level'].value_counts().plot(
    kind='bar', ax=axes[1],
    color=[risk_colors.get(r, '#BDC3C7') for r in df['Risk Level'].value_counts().index],
    edgecolor='white'
)
axes[1].set_title('Risk Level Distribution')
axes[1].set_xlabel('Risk Level')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 8. Prepare Feature Matrix & Target

In [ ]:
# Drop rows where WQI could not be calculated
valid_idx = df[df['Risk Level'] != 'Unknown'].index
X = df[FEATURES].loc[valid_idx].reset_index(drop=True)
y = df['Risk Level'].loc[valid_idx].reset_index(drop=True)

print(f"Rows removed (unknown WQI): {len(df) - len(X)}")
print(f"Training set shape: {X.shape}")
print(f"\nRisk Level distribution:")
print(y.value_counts())

## 9. Imputation
Using median strategy — appropriate for skewed water chemistry data.

In [ ]:
print("Missing values before imputation:")
print(X.isnull().sum())
print(f"Total missing cells: {X.isnull().sum().sum()}")

In [ ]:
imputer   = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)

print("Missing values after imputation:")
print(X_imputed.isnull().sum())
print(f"Total missing: {X_imputed.isnull().sum().sum()}")

print("\nMedian values learned by imputer:")
for col, val in zip(FEATURES, imputer.statistics_):
    print(f"  {col}: {val:.4f}")

## 10. Encode Target Labels

In [ ]:
encoder   = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Label mapping:")
for label, code_val in zip(encoder.classes_, range(len(encoder.classes_))):
    print(f"  {label} → {code_val}")

print(f"\nClass distribution after encoding:")
for code_val, label in enumerate(encoder.classes_):
    count = (y_encoded == code_val).sum()
    print(f"  {code_val} ({label}): {count} samples")

## 11. Class Imbalance Check & SMOTE

In [ ]:
print("Class distribution before SMOTE:")
for code_val, label in enumerate(encoder.classes_):
    print(f"  {label}: {(y_encoded == code_val).sum()}")

# SMOTE configured to oversample minority classes only
smote = SMOTE(
    sampling_strategy='not majority',
    k_neighbors=3,
    random_state=42
)
print("\nSMOTE configured — will oversample all classes except the majority.")

## 12. Build Pipeline & Cross-Validate

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

rf_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  smote),
    ('model',  rf_model),
])

print("Pipeline built:")
print("  StandardScaler → SMOTE → RandomForestClassifier")
print(f"  n_estimators: 200  |  max_depth: 6  |  min_samples_leaf: 4")

In [ ]:
rf_scores = cross_val_score(
    rf_pipeline, X_imputed, y_encoded,
    cv=cv, scoring='accuracy'
)

print("5-Fold Cross-Validation Results:")
print(f"  Fold scores : {rf_scores.round(4)}")
print(f"  Mean accuracy: {rf_scores.mean():.4f}")
print(f"  Std deviation: {rf_scores.std():.4f}")

## 13. Train on Full Dataset

In [ ]:
rf_pipeline.fit(X_imputed, y_encoded)
print("Model trained on full dataset.")

## 14. Evaluate — Confusion Matrix & Classification Report

In [ ]:
y_pred = cross_val_predict(rf_pipeline, X_imputed, y_encoded, cv=cv)
labels = encoder.classes_

# Confusion matrix
cm = confusion_matrix(y_encoded, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels
)
plt.title('Random Forest — Confusion Matrix (5-Fold CV)')
plt.ylabel('Actual Risk Level')
plt.xlabel('Predicted Risk Level')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_encoded, y_pred, target_names=labels))

## 15. Feature Importance

In [ ]:
importances = rf_pipeline.named_steps['model'].feature_importances_
feat_imp    = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#E74C3C' if v == feat_imp.max() else '#3498DB' for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors, edgecolor='white')
plt.title('Random Forest — Feature Importance (6-Parameter Model)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Feature Importances:")
for feat, imp in feat_imp.sort_values(ascending=False).items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:<20} {bar}  {imp:.4f}")

## 16. Save Artifacts
Saves to `model/artifacts/` — same path expected by `processor.py`.

In [ ]:
os.makedirs('model/artifacts', exist_ok=True)

joblib.dump(rf_pipeline, 'model/artifacts/model.pkl')
joblib.dump(imputer,     'model/artifacts/imputer.pkl')
joblib.dump(encoder,     'model/artifacts/label_encoder.pkl')

import json
meta = {
    "features":            FEATURES,
    "label_names":         {str(i): label for i, label in enumerate(encoder.classes_)},
    "feature_importances": dict(zip(FEATURES, importances.tolist())),
    "cv_accuracy":         float(rf_scores.mean()),
    "cv_std":              float(rf_scores.std()),
    "n_samples":           int(len(X_imputed)),
    "who_limits":          {k: v['max'] for k, v in LIMITS.items() if v['max'] is not None},
    "turbidity_note":      "Turbidity is accepted from frontend at inference but set to 0.0 during training (not in dataset).",
}
with open('model/artifacts/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Artifacts saved:")
print("  model/artifacts/model.pkl")
print("  model/artifacts/imputer.pkl")
print("  model/artifacts/label_encoder.pkl")
print("  model/artifacts/model_meta.json")

## 17. Verify Saved Artifacts
Confirm the saved model loads correctly and produces valid predictions.

In [ ]:
# Load and verify
model_loaded   = joblib.load('model/artifacts/model.pkl')
imputer_loaded = joblib.load('model/artifacts/imputer.pkl')
encoder_loaded = joblib.load('model/artifacts/label_encoder.pkl')

# Run a test prediction
test_sample = np.array([[6.2, 5.0, 4.8, 45.0, 0.2, 1800]])
test_imputed = imputer_loaded.transform(test_sample)
pred_code    = model_loaded.predict(test_imputed)[0]
pred_label   = encoder_loaded.inverse_transform([pred_code])[0]
pred_conf    = round(model_loaded.predict_proba(test_imputed)[0][pred_code] * 100, 1)

print("Verification — test prediction:")
print(f"  Input : ph=6.2, turbidity=5.0, dissolved_oxygen=4.8,")
print(f"          nitrates=45.0, phosphates=0.2, salinity=1800")
print(f"  Output: {pred_label} ({pred_conf}% confidence)")
print(f"\nArtifacts verified ✅ — ready to deploy.")

## 18. Download Artifacts
Run the cell below to download all three `.pkl` files from Colab to your machine.

In [ ]:
from google.colab import files

files.download('model/artifacts/model.pkl')
files.download('model/artifacts/imputer.pkl')
files.download('model/artifacts/label_encoder.pkl')
files.download('model/artifacts/model_meta.json')

print("Download started for all 4 artifact files.")
print("Place them in your project at:  model/artifacts/")